In [1]:
import os, subprocess, sys, time, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from adaptive_behaviour import paths

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

OUT = paths.RESULTS_BASELINES; OUT.mkdir(parents=True, exist_ok=True)
RUNNER = paths.RUNNERS / "run_baseline.py"
assert RUNNER.exists(), f"baseline runner not found at {RUNNER}"

N_SEEDS   = 99
N_WORKERS = 6
SEEDS     = list(range(N_SEEDS))
AGENTS    = ["Random", "Heuristic", "DynaQ", "DynaQ-replay"]

# Pin BLAS to one thread per worker. Each runner otherwise spawns one BLAS thread per core,
# so N concurrent runners create N x 12 threads on a 12-core machine. With 2432-state matrices
# threaded BLAS gives no speed-up, and the contention dominated everything on the first attempt
# (load average 55 on 12 cores).
BLAS_ENV = dict(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1",
                VECLIB_MAXIMUM_THREADS="1", NUMEXPR_NUM_THREADS="1")

# Measured single-threaded and uncontended: Random ~3 s/seed, DynaQ ~6 s, Heuristic ~38 s.
print(f"{len(AGENTS)} baselines x {N_SEEDS} seeds, {N_WORKERS} worker processes")
print(f"outputs -> {paths.rel(OUT)}")

4 baselines x 99 seeds, 6 worker processes
outputs -> <repo>/main/pong_trials/baselines


In [2]:
def path_for(agent, seed):
    return OUT / f"{agent}_{seed}.csv"


def run_one(job):
    """Each run gets its OWN PROCESS.

    The environment keeps module-level state (`readState.games`) and `revision_runners.get_env`
    returns a single shared instance, so threads interleave episodes and corrupt every run
    involved. An earlier threaded version of this cell crashed inside `end_rally` for exactly
    that reason.
    """
    agent, seed = job
    p = path_for(agent, seed)
    if p.exists():
        return agent, seed, 0.0, 0
    env = dict(os.environ, **BLAS_ENV)
    t0 = time.time()
    r = subprocess.run([sys.executable, str(RUNNER), agent, str(seed)],
                       capture_output=True, text=True, cwd=str(paths.RUNNERS), env=env)
    return agent, seed, time.time() - t0, r.returncode


jobs = [(a, s) for a in AGENTS for s in SEEDS if not path_for(a, s).exists()]
print(f"{len(AGENTS) * len(SEEDS) - len(jobs)} already done; running {len(jobs)}\n")

failures = []
if jobs:
    t_start = time.time()
    done_count = 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:   # threads only shepherd subprocesses
        futures = [ex.submit(run_one, j) for j in jobs]
        for fut in as_completed(futures):
            agent, seed, dt, rc = fut.result()
            done_count += 1
            if rc:
                failures.append((agent, seed))
            if done_count % 20 == 0 or done_count == len(jobs):
                print(f"[{done_count:>4}/{len(jobs)}] latest {agent} seed {seed} "
                      f"({dt:.1f}s)   elapsed {(time.time()-t_start)/60:.1f} min")
    print(f"\ntotal: {(time.time()-t_start)/60:.1f} min")
    if failures:
        print(f"{len(failures)} FAILED: {failures[:10]}")

have = {a: sum(path_for(a, s).exists() for s in SEEDS) for a in AGENTS}
print("\nseeds complete:", have)

260 already done; running 136

[  20/136] latest DynaQ seed 47 (14.9s)   elapsed 4.0 min
[  40/136] latest DynaQ seed 95 (29.8s)   elapsed 5.8 min
[  60/136] latest DynaQ-replay seed 17 (49.8s)   elapsed 8.2 min
[  80/136] latest DynaQ-replay seed 37 (50.7s)   elapsed 10.8 min
[ 100/136] latest DynaQ-replay seed 60 (38.2s)   elapsed 13.4 min
[ 120/136] latest DynaQ-replay seed 81 (37.6s)   elapsed 15.9 min
[ 136/136] latest DynaQ-replay seed 98 (47.3s)   elapsed 18.0 min

total: 18.0 min

seeds complete: {'Random': 99, 'Heuristic': 99, 'DynaQ': 99, 'DynaQ-replay': 99}


In [ ]:
from adaptive_behaviour.pong.analysis.tables import baseline_tables as B

# The analysis lives in baseline_tables.py so that this notebook and the paper tables cannot
# drift apart. It reproduces 01_statistics.ipynb exactly: same window definition, same derived
# metrics, same replicate-level cluster bootstrap, same base seed.
ep = B.load_episodes(".")
sess = B.session_means(ep)

# Regression check. If the published groups no longer reproduce Supplementary Table tab:absolute
# to the printed digit, something upstream has changed and the new rows cannot be trusted either.
_pub = B.absolute_table(sess, ["MCC", "HCC", "CFL-1", "CFL-2", "CFL-3", "CFL-4",
                               "AIF-1", "DP-2", "DP-5", "DP-10", "DP-20"])
_expected = {"MCC": (0.85, 0.81, 0.91), "HCC": (0.88, 0.82, 0.95), "CFL-1": (1.02, 0.90, 1.18),
             "CFL-2": (1.01, 0.92, 1.11), "CFL-3": (1.44, 1.26, 1.64), "CFL-4": (2.47, 1.80, 3.59),
             "AIF-1": (0.80, 0.77, 0.84), "DP-2": (0.96, 0.86, 1.07), "DP-5": (1.19, 1.06, 1.33),
             "DP-10": (1.18, 1.05, 1.32), "DP-20": (0.98, 0.89, 1.08)}
for _, r in _pub.iterrows():
    got = (round(r.rally, 2), round(r.rally_lo, 2), round(r.rally_hi, 2))
    assert got == _expected[r.group], f"{r.group}: {got} != {_expected[r.group]}"
print("published groups reproduce tab:absolute exactly\n")

within = B.within_table(sess)
print("Average rally length by window, per-seed means (the convention used throughout the paper)")
print(within[["group", "source", "n", "w1", "w2", "gain", "lo", "hi", "p_adj"]]
      .sort_values("w2", ascending=False).round(3).to_string(index=False))